# Paired A01/B01 skill forecasting

An analogue of `09_skill_forecasting.ipynb`, using **both aircraft's observation streams**
to predict **A01's selected catalogue skill** at **+1, +2 and +5 seconds**. The target
remains A01; B01 is a scripted controller, not a second learned catalogue target.

**Run All downloads the required LFS files, prepares data and trains for up to 100 epochs.
Early stopping is enabled by default and can be switched off in the configuration cell.**
Default CUDA acceleration uses **BF16**, **512-window training batches**, **1024-window
evaluation batches**, and a **normalized GPU cache of unique frames**. CPU execution
falls back to FP32 and direct memory-mapped batches. A prefetched CPU data path and
`torch.compile` are optional. The configuration cell exposes all switches. The
new run directory preserves earlier experiments; resume requires the same source
and configuration. It produces inline train/test per-class metrics every epoch and final inline diagnostics
for train, test and validation. No trained results are bundled with this notebook.
Dependencies: Python 3.10+, NumPy, pandas, PyArrow, PyTorch 2.1+, Matplotlib and IPython.
For example, run `%pip install -e ".[ml]"` from the repository root in your notebook environment.
The reported training setup is an **NVIDIA GeForce RTX 5080** (assuming its
standard **16 GB VRAM**) with **PyTorch 2.11.0+cu130** and CUDA 13.1 reported.
`+cu130` identifies a CUDA **13.0** PyTorch build; a driver supporting CUDA 13.1
can run it. These version numbers do not need to match exactly, and no reinstall
is indicated solely by that difference. The runtime table records the actual
PyTorch CUDA runtime, GPU architecture, BF16 availability and a small CUDA kernel check.

The reported CPU is **GenuineIntel, Family 6 Model 186 Stepping 2**, with
**10 physical cores and 16 logical processors**, architecture **AMD64/x86-64**.
AMD64 is the instruction-set name here; the CPU vendor is Intel. The reported
2,500 MHz frequency and 15.5% utilization are snapshots, not throughput benchmarks.
Reported target hardware is shown separately from the environment executing the
notebook, so CPU-only checks performed elsewhere are not attributed to this machine. Prepared frames use about 1.1 GB; allow several GB for
raw data, arrays, checkpoints and reports. Histories are gathered from a memory map.

The reference notebook and configuration were inspected at repository revision
`b5e8bb13dda92f9e1070b0cc3f797a7ca8028c60`.
This notebook's companion module is `bvr_behavior_prediction/paired_skill_forecasting.py`.

**MLflow tracking is enabled by default.** Training runs receive unique timestamped
folders and are grouped in the `paired-skill-forecasting` experiment. The dedicated
SQLite store and model/figure artifacts live under `artifacts/paired_skill_forecasting/`.
Set `MLFLOW_TRACKING_URI` for a remote server, or `MLFLOW_ENABLED=False` to disable tracking.
The setup cell prints the command to start the local comparison UI. Resumption uses
`EXISTING_RUN` and reuses the saved MLflow run ID.

Every executable cell starts with IPython's `%%time` magic. Its output reports
**CPU time and elapsed wall time** after the cell finishes, including the setup
cell itself. CUDA work is synchronized before each cell finishes so GPU kernels
are included in elapsed time. Markdown cells contain no executable work and are
not timed. Use an IPython/Jupyter kernel to execute these cells.

On Windows, the notebook kernel must be able to launch Git. The download helper
checks PATH and standard Git for Windows locations. If needed, set
`GIT_EXECUTABLE = r"C:\Program Files\Git\cmd\git.exe"` in the setup cell.
For a Conda-managed install, run `conda install -c conda-forge git git-lfs` in
the notebook's environment, then restart Jupyter so it inherits the updated PATH.
If the raw corpus Parquet files have already been downloaded, set `DOWNLOAD_LFS=False`.


In [ ]:
%%time
from pathlib import Path
import json
import os
import platform
import subprocess
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display, Markdown

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "bvr_behavior_prediction/paired_skill_forecasting.py").exists())
sys.path.insert(0, str(ROOT))
from bvr_behavior_prediction.paired_skill_forecasting import (
    DEFAULT_TRAINING, FEATURES, HORIZONS, HISTORY_STEPS, RAW_HISTORY_STEPS, SPLITS,
    PairedData, prepare_dataset, train_model, final_evaluation, metric_tables,
    read_json, sha256, resolve_precision, benchmark_batches, download_paired_corpora,
)

DATA = ROOT / "artifacts/paired_skill_forecasting/data_40_30_30_v1"
RUN_LABEL = "paired-bf16-cached"
EXISTING_RUN = None  # Set to an existing run directory for RESUME=True or TRAIN_MODEL=False.
RUN = Path(EXISTING_RUN).expanduser().resolve() if EXISTING_RUN else ROOT / "artifacts/paired_skill_forecasting/runs" / (RUN_LABEL + "_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ"))
DOWNLOAD_LFS = True
GIT_EXECUTABLE = None  # Or r"C:\Program Files\Git\cmd\git.exe" if Git is absent from the kernel PATH.
PREPARE_DATA = True
TRAIN_MODEL = True
RESUME = False  # Resume the same run; completed/early-stopped runs do not train again.
EARLY_STOPPING = True  # False always runs the full 100 epochs.
if RESUME and EXISTING_RUN is None:
    raise ValueError("Set EXISTING_RUN to the previous run directory before resuming.")
if not TRAIN_MODEL and EXISTING_RUN is None:
    raise ValueError("Set EXISTING_RUN to the completed run directory for a report-only execution.")
MLFLOW_ENABLED = True
MLFLOW_EXPERIMENT = "paired-skill-forecasting"
MLFLOW_DIRECTORY = ROOT / "artifacts/paired_skill_forecasting"
MLFLOW_DIRECTORY.mkdir(parents=True, exist_ok=True)
MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", f"sqlite:///{(MLFLOW_DIRECTORY / 'mlflow.db').as_posix()}")
MLFLOW_ARTIFACT_ROOT = (MLFLOW_DIRECTORY / "mlflow_artifacts").as_uri()
TRACKING = {"enabled": MLFLOW_ENABLED, "tracking_uri": MLFLOW_TRACKING_URI,
            "experiment": MLFLOW_EXPERIMENT, "run_name": RUN.name,
            "artifact_root": MLFLOW_ARTIFACT_ROOT if MLFLOW_TRACKING_URI.startswith("sqlite:") else None,
            "log_per_class": True, "log_model": True}
from bvr_behavior_prediction.paired_skill_tracking import log_run_artifacts, compare_runs, metric_history
print("Run directory:", RUN)
if MLFLOW_ENABLED and MLFLOW_TRACKING_URI.startswith("sqlite:"):
    print(f'mlflow ui --backend-store-uri "{MLFLOW_TRACKING_URI}" --host 127.0.0.1 --port 5000')
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PRECISION = "bf16"           # CUDA BF16 autocast; CPU/unsupported devices use FP32.
DATA_BACKEND = "gpu_cache"   # Alternatives: "prefetch" (CPU workers), "direct".
BATCH_SIZE = 512             # Benchmark 512 and 1024 below; choose explicitly.
EVALUATION_BATCH_SIZE = 1024
CPU_THREADS = 4
NUM_WORKERS = 2              # Used only by DATA_BACKEND="prefetch"; try 2, then 4.
PREFETCH_FACTOR = 2
COMPILE_MODEL = False        # Opt-in torch.compile; needs a supported Inductor/Triton setup.
COMPILE_MODE = "default"     # Also supports "reduce-overhead" / "max-autotune".
RUN_BATCH_BENCHMARK = DEVICE == "cuda"
BENCHMARK_COMPILATION = False  # True compares eager/compiled; first compilation can be slow.
CONFIG = dict(DEFAULT_TRAINING)
CONFIG.update(early_stopping=EARLY_STOPPING,
              early_stopping_min_epochs=50, early_stopping_patience=10,
              precision=PRECISION, data_backend=DATA_BACKEND,
              batch_size=BATCH_SIZE, evaluation_batch_size=EVALUATION_BATCH_SIZE,
              cpu_threads=CPU_THREADS, num_workers=NUM_WORKERS, prefetch_factor=PREFETCH_FACTOR,
              compile_model=COMPILE_MODEL, compile_mode=COMPILE_MODE)
reported_hardware = {
    "PyTorch": "2.11.0+cu130", "PyTorch CUDA runtime": "13.0",
    "Reported CUDA/driver capability": "13.1", "GPU": "NVIDIA GeForce RTX 5080",
    "GPU memory assumption": "16 GB", "CPU": "Intel64 Family 6 Model 186 Stepping 2, GenuineIntel",
    "Architecture": "AMD64 / x86-64", "Physical cores": 10, "Logical processors": 16,
    "Frequency snapshot": "2500 MHz current/max", "CPU utilization snapshot": "15.5%",
}
display(pd.Series(reported_hardware, name="Reported target training machine"))
runtime = {"PyTorch": str(torch.__version__), "PyTorch CUDA runtime": torch.version.cuda,
           "Selected device": DEVICE, "CPU identifier": platform.processor(),
           "Architecture": platform.machine(), "Logical processors": os.cpu_count(),
           "PyTorch intra-op threads now": torch.get_num_threads(),
           "PyTorch inter-op threads now": torch.get_num_interop_threads(),
           "Requested precision": PRECISION, "Effective precision": resolve_precision(DEVICE, PRECISION),
           "Requested data backend": DATA_BACKEND,
           "Fitting loop intra-op threads": min(CPU_THREADS, os.cpu_count() or 1)}
if DEVICE == "cuda":
    properties = torch.cuda.get_device_properties(0)
    runtime.update({"GPU": properties.name, "GPU memory (GiB)": properties.total_memory / 2**30,
                    "Compute capability": torch.cuda.get_device_capability(0),
                    "Compiled CUDA architectures": torch.cuda.get_arch_list(),
                    "BF16 supported": torch.cuda.is_bf16_supported()})
    # A driver/device query alone does not prove that the installed kernels execute.
    cuda_probe = torch.ones((16, 16), device="cuda")
    cuda_probe_result = cuda_probe @ cuda_probe
    torch.cuda.synchronize()
    runtime["CUDA kernel smoke check"] = "passed"
    del cuda_probe, cuda_probe_result
display(pd.Series(runtime, name="Detected notebook runtime"))
assert CONFIG["epochs"] == 100
assert CONFIG["heads"] == 4 * 1.5 == 6
assert CONFIG["width"] % CONFIG["heads"] == 0
display(pd.Series({**CONFIG, "device": DEVICE, "history_steps": HISTORY_STEPS,
                   "raw_history_steps": RAW_HISTORY_STEPS, "features": len(FEATURES),
                   "horizons_s": HORIZONS}, name="Experiment configuration"))

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


## Experimental contract

* **History:** 51 chronological frames at 10 Hz span five seconds. Five extra past
  frames support a backward 0.5-second acceleration estimate. The first origin is
  six seconds, then every second, matching notebook 09. No histories cross episodes.
* **Inputs:** A01's 51 observations plus three backward acceleration features,
  concatenated with the equivalent 54 B01 features: **108 features per frame**.
  Both viewpoints must be available at inference time. This assumes shared telemetry
  or centralized observation access and changes the reference's B01-only input contract.
  Future observations, decisions, pilot IDs, checkpoint IDs and outcomes are excluded.
* **Targets:** the actual A01 skill recorded at each exact future timestamp.
  Missing future decisions are masked separately; there is no forward fill. These
  are catalogue selections, not necessarily distinct executed physical manoeuvres.
* **Architecture:** two Transformer layers, **six attention heads**, dropout 0.1.
  Width increases from 128 to **192** to preserve 32 dimensions per head and allow
  six heads. This changes capacity as well as head count; an ablation is needed to
  attribute any performance change to the added viewpoint or attention heads.
* **Split:** **train/test/validation = 0.4/0.3/0.3**, stratified by starting range
  and grouped by scenario. For the full corpora: **120/90/90 groups**, or
  **1,200/900/900 encounters**. All ten pilots and both viewpoints for a scenario
  stay together. Window counts need not follow these ratios because durations vary.
* **Selection:** keep every epoch whose mean **test** NLL strictly improves over
  all previous epochs. The test partition is consequently a model-selection set;
  its final score is selection-biased. Validation is evaluated only after selection
  and provides the held-out estimate. Do not tune against it afterward.
* **Acceleration:** CUDA BF16 autocast for network forward passes; FP32 model weights,
  optimizer state, cross-entropy, probabilities and reporting. Unique frames are
  normalized once on GPU with training-only statistics; labels/anchors are cached
  alongside them. No expanded-window cache is created. Default batches are 512
  for fitting and 1024 for evaluation; epoch batch tails remain included.
* **Training:** up to 100 epochs, with optional early stopping on mean test NLL.
  The first 50 epochs always run; patience counting starts at epoch 51. Ten
  consecutive epochs without a new all-time minimum stop training, so epoch 60
  is the earliest stop. A strict improvement resets the counter. Equal-weight
  mean of the three available horizon cross-entropies, AdamW, gradient clipping,
  train-only scaling.

In [ ]:
%%time
# Download only the required arrays, not videos, transfer tests or reproduction ZIPs.
includes = [
    "shared/corpora/tactical-episodes-v1/episodes.parquet",
    "shared/corpora/tactical-episodes-v1/observations/*.parquet",
    "shared/corpora/trained-pilot-episodes-v1/episodes.parquet",
    "shared/corpora/trained-pilot-episodes-v1/observations/*.parquet",
    "shared/corpora/trained-pilot-episodes-v1/tracks/*.parquet",
    "shared/corpora/trained-pilot-episodes-v1/target_decisions/*.parquet",
]
if DOWNLOAD_LFS and not (DATA / "manifest.json").exists():
    download_paired_corpora(ROOT, includes, git_executable=GIT_EXECUTABLE)

if PREPARE_DATA and not (DATA / "manifest.json").exists():
    prepare_dataset(ROOT, DATA)
assert (DATA / "manifest.json").exists(), "Prepare data first or enable PREPARE_DATA."
data = PairedData(DATA)  # Verifies prepared-file hashes and scenario isolation.
assert data.manifest["split_fractions"] == {"train": .4, "test": .3, "validation": .3}
assert data.manifest["history_steps"] == 51 and data.manifest["raw_history_steps"] == 56
assert data.manifest["features"] == FEATURES
assert data.manifest["horizons_s"] == [1, 2, 5]
for relative, expected in data.manifest["source_manifests"].items():
    assert sha256(ROOT / relative / "manifest.json") == expected, "Raw corpus provenance changed."
display(pd.DataFrame(data.manifest["counts"]).T)
display(Markdown(f"**Catalogue:** {len(data.classes)} classes. **Device:** {DEVICE}."))

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


## Split and label-support audit

Preparation verifies source checksums, matched episode IDs, policies, scenario seeds,
frame counts, timestamps, and opposing views of the same aircraft positions and velocities.
Divergent reruns fail instead of being concatenated. Normalization uses the union of
training-history frames only. All three horizons use independent missing-label masks.

The batching and memory table below is calculated from the prepared dataset and
current configuration. Batches contain forecast windows, not whole encounters.
On CUDA, the default `gpu_cache` path copies unique frames in bounded chunks and
normalizes them once with train-only statistics; each batch is gathered on GPU.
The CPU memory map remains unchanged. `prefetch` instead uses a single reusable
worker pool with vectorized batches, pinned memory and a separate CUDA copy stream.
Workers reopen memory maps rather than serializing the dataset (including on
Windows/spawn). Use `NUM_WORKERS=0` to debug that path. `direct` retains the original
CPU gathering path. CPU execution automatically substitutes `direct` for `gpu_cache`.


In [3]:
%%time
assignments = pd.read_csv(DATA / "split_assignments.csv")
display(assignments.groupby("split").agg(
    scenarios=("scenario_group", "nunique"), episodes=("episode_id", "nunique")))
support_rows = []
for split in SPLITS:
    for h, horizon in enumerate(HORIZONS):
        labels = data.targets[data.indices[split], h]
        counts = np.bincount(labels[labels >= 0], minlength=len(data.classes))
        support_rows.extend({"split": split, "horizon_s": horizon, "skill": name,
                             "support": int(count)} for name, count in zip(data.classes, counts))
support = pd.DataFrame(support_rows)
with pd.option_context("display.max_rows", None):
    display(support.pivot(index=["horizon_s", "skill"], columns="split", values="support"))

# Resource estimates use configured shapes; no additional fitting is performed.
import math
from bvr_behavior_prediction.paired_skill_forecasting import PairedTransformer

with torch.random.fork_rng(devices=[]):
    resource_model = PairedTransformer(num_classes=len(data.classes), **{
        k: CONFIG[k] for k in ("width", "heads", "layers", "dropout")})
    parameter_count = sum(p.numel() for p in resource_model.parameters())
del resource_model

window_bytes = HISTORY_STEPS * len(FEATURES) * np.dtype(np.float32).itemsize
resource_rows = [
    ("Parameters", parameter_count, "parameters"),
    ("FP32 model weights", parameter_count * 4 / 2**20, "MiB"),
    ("FP32 weights + gradients + two Adam states", parameter_count * 16 / 2**20, "MiB"),
    ("Raw feature memory-map file", data.features.nbytes / 2**30, "GiB (not necessarily RSS)"),
    ("GPU frame/anchor/target cache estimate", (data.features.nbytes + data.anchors.nbytes
                                             + data.targets.nbytes) / 2**30, "GiB"),
    ("Expanded windows if all were cached", len(data.targets) * window_bytes / 2**30, "GiB"),
    ("Training batch input", CONFIG["batch_size"] * window_bytes / 2**20, "MiB"),
    ("Evaluation batch input", CONFIG["evaluation_batch_size"] * window_bytes / 2**20, "MiB"),
    ("Optimizer batches per epoch", math.ceil(len(data.indices["train"]) / CONFIG["batch_size"]), "batches"),
]
for split in ("train", "test"):
    count = len(data.indices[split])
    resource_rows.append((f"{split} evaluation batches per epoch",
                          math.ceil(count / CONFIG["evaluation_batch_size"]), "batches"))
    resource_rows.append((f"{split} evaluation probabilities",
                          count * len(HORIZONS) * len(data.classes) * 4 / 2**20, "MiB"))
display(pd.DataFrame(resource_rows, columns=["Quantity", "Value", "Unit"]).round(3))
print("Model activations, framework workspaces, allocator reserves, data-page cache and")
print("temporary arrays are additional. Measure CUDA peak allocated/reserved memory on the target GPU.")

# Disposable training-only benchmark: no test/validation access and no run/checkpoint writes.
# RNG states are restored afterward. This does not change BATCH_SIZE automatically.
if RUN_BATCH_BENCHMARK:
    batch_benchmark = benchmark_batches(data, CONFIG, DEVICE, batch_sizes=(512, 1024),
                                        compare_compilation=BENCHMARK_COMPILATION)
    display(batch_benchmark.round(4))
    benchmark_dir = ROOT / "artifacts/paired_skill_forecasting/benchmarks"
    benchmark_dir.mkdir(parents=True, exist_ok=True)
    benchmark_path = benchmark_dir / ("batches_" + pd.Timestamp.now(tz="UTC").strftime("%Y%m%dT%H%M%S%fZ") + ".csv")
    batch_benchmark.to_csv(benchmark_path, index=False)
    print("Saved benchmark:", benchmark_path)
    TRACKING["benchmark_path"] = str(benchmark_path)
else:
    print("Batch benchmark skipped. Enable RUN_BATCH_BENCHMARK to run it explicitly.")

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


,scenarios,episodes
split,,
test,90,900
train,120,1200
validation,90,900


split                           test  train  validation
horizon_s skill                                        
1         abort_support          368    355         266
          accelerate_to_speed      0      0           3
          beam_missile_left       22     25          18
          beam_missile_right      28     42          30
          beam_target_left      3890   5858        4246
          beam_target_right     2208   4135        3047
          climb_to_altitude        3      0           1
          commit                   0      0           0
          crank_target_left     4057   5245        3523
          crank_target_right    1700   2291        1869
          decelerate_to_speed    255    466         278
          defensive_reversal       6      3           1
          descend_to_altitude    269    841         470
          dive_defense             0      0           0
          drag_missile             0      0           0
          extend                5140   7427        5491
          lag_pursuit          11122  15252       10638
          last_ditch_break         3      2           1
          launch                 275    295         355
          lead_pursuit          3445   4125        2861
          lock_target              3      5           0
          maintain_heading        59     43          53
          notch_left             320    571         268
          notch_right            192    219         127
          preemptive_crank         1      4           1
          pursue_target         3757   6002        5497
          recommit              8642  12131        9475
          search                  12     21          24
          secondary_shot        3282   4104        2867
          short_range_attack     308    497         457
          support_missile         59    165          85
          turn_cold            11478  15415       11017
          turn_to_heading         85    162          57
2         abort_support          378    366         270
          accelerate_to_speed      0      0           3
          beam_missile_left       23     26          18
          beam_missile_right      32     47          30
          beam_target_left      3720   5615        4061
          beam_target_right     2237   4168        3074
          climb_to_altitude        3      0           1
          commit                   0      0           0
          crank_target_left     3716   4831        3201
          crank_target_right    1708   2303        1874
          decelerate_to_speed    261    466         278
          defensive_reversal       6      4           1
          descend_to_altitude    272    810         459
          dive_defense             0      0           0
          drag_missile             0      0           0
          extend                5131   7406        5467
          lag_pursuit          11076  15188       10604
          last_ditch_break         3      2           1
          launch                 277    294         351
          lead_pursuit          3428   4113        2854
          lock_target              3      5           0
          maintain_heading        61     48          58
          notch_left             317    558         263
          notch_right            195    226         131
          preemptive_crank         1      4           1
          pursue_target         3718   5948        5452
          recommit              8629  12096        9447
          search                  12     23          24
          secondary_shot        3264   4048        2814
          short_range_attack     322    502         459
          support_missile         57    157          82
          turn_cold            11434  15380       10998
          turn_to_heading         85    160          56
5         abort_support          391    397         265
          accelerate_to_speed      0      0           3
          beam_missile_left       19     20          12
          beam_missile_r

,Quantity,Value,Unit
0,Parameters,976995.000,parameters
1,FP32 model weights,3.727,MiB
2,FP32 weights + gradients + two Adam states,14.908,MiB
3,Raw feature memory-map file,1.028,GiB (not necessarily RSS)
4,GPU frame/anchor/target cache estimate,1.034,GiB
5,Expanded windows if all were cached,4.303,GiB
6,Training batch input,10.758,MiB
7,Evaluation batch input,21.516,MiB
8,Optimizer batches per epoch,168.000,batches
9,train evaluation batches per epoch,84.000,batches


Model activations, framework workspaces, allocator reserves, data-page cache and
temporary arrays are additional. Measure CUDA peak allocated/reserved memory on the target GPU.
Batch benchmark skipped. Enable RUN_BATCH_BENCHMARK to run it explicitly.
CPU times: user 62.9 ms, sys: 3.47 ms, total: 66.4 ms
Wall time: 42.7 ms


## Train for up to 100 epochs

Each epoch ends with a fresh **evaluation-mode** pass over train and test, using
the same updated checkpoint, without dropout. Each inline table contains per-class
F1, support and **class-conditional NLL** (mean cross-entropy among examples whose
true label is that class) at all three horizons. Unsupported classes have zero F1
and undefined loss. Macro-F1 includes all catalogue classes, with undefined F1 set
to zero, matching notebook 09. Supported macro-F1 is also recorded.

Outputs are retained for every epoch; there is no clearing of previous output.
`epochs/` stores full CSVs; `checkpoints/epoch_NNN.pt` retains **every new minimum**
of mean test NLL. `best.pt` selects the lowest-loss epoch and `last.pt` supports
resumption, including optimizer, RNG and early-stopping counter states.
With `EARLY_STOPPING=True`, counting begins after epoch 50; ten consecutive
non-improving test-loss epochs stop training. Set it to `False` for all 100 epochs.
`training_status.json` records whether training completed or stopped early.
Resuming an early-stopped run returns its history without fitting more epochs. A nonempty run directory requires
`RESUME=True` or a fresh `RUN` path. GPU/CPU floating-point results can differ. Effective precision, backend, CPU
thread count and cache size are recorded in `run.json`. Changing precision, batch
size, compilation or data backend requires a fresh run directory.

The batch benchmark uses disposable models and **training data only**, preserves
RNG state, includes gathering/transfers/forward/backward/AdamW in each step, and
reports measured throughput plus peak allocated/reserved VRAM. Cold/warmup time
is separate. It never adjusts the learning rate or selects a checkpoint. Set
`BATCH_SIZE=1024` only after comparing the benchmark and memory headroom; larger
batches change the number of optimizer updates per epoch. `BENCHMARK_COMPILATION=True`
compares eager and compiled variants. Compilation is optional and can require
additional backend support on Windows; failures should be resolved or compilation
disabled, rather than assuming an unmeasured speedup. Checkpoints always save
unwrapped FP32 weights and can be loaded with or without compilation.

The `%%time` output at the bottom of the fitting cell measures the entire fitting
cell, including checkpoint writes and inline rendering. The existing per-epoch
`seconds` field measures fitting and train/test evaluation before checkpoint/table
and MLflow I/O; it should not be mistaken for total per-epoch wall time. A phase profiler is
recommended before changing the implementation for throughput.

### MLflow run/model comparison

Configuration, dataset/source hashes and actual runtime precision/backend are tracked.
Per-epoch train/test metrics are logged both as horizon means and by horizon; per-class
F1, class-conditional NLL and support are included by default. Unsupported-class NLL
is omitted rather than logged as zero. Set `TRACKING["log_per_class"]=False` to reduce
metric logging; the complete per-class CSV artifacts are still retained.

Every local improving checkpoint remains available. MLflow stores the selected
`best.pt`, all epoch/final metric CSVs, source/data metadata, and final PNG figures;
it does not copy the large raw corpora, prepared feature arrays or prediction NPZs.
The selected network is also exported as a standard **MLflow PyTorch model** at
`runs:/<run_id>/model`, with a signature, packaged source, normalization statistics,
feature ordering, classes and horizons. It accepts **already-normalized (B,51,108)
feature histories** and returns **(B,3,C) logits**. Weights are FP32; use the recorded
CUDA autocast precision when reproducing BF16 evaluation. No model registry deployment
or serving endpoint is created.

Use the MLflow UI to select runs and compare parameters/metrics. The final notebook
section provides a comparison table and loss curves filtered to the **same prepared
dataset hash**. Test loss selects checkpoints; validation remains a final descriptive
score. A new experiment needs a new untouched evaluation set if those validation
scores are used to guide further tuning.

A failed/interrupted attempt is marked FAILED/KILLED; resumption reuses the run ID
and backfills any completed local epochs not yet tracked. Model checkpoints remain
portable and local even if an MLflow write fails. Tracking uses a client scoped to
the specified URI and does not replace an unrelated fluent MLflow active run.



In [ ]:
%%time
def epoch_output(row, summary, per_class):
    marker = " — saved improving checkpoint" if row["improved"] else ""
    display(Markdown(f"### Epoch {row['epoch']:03d}/{CONFIG['epochs']}{marker}"))
    display(summary[["split", "horizon_s", "nll", "accuracy", "macro_f1",
                     "supported_macro_f1", "weighted_f1"]].round(4))
    table = per_class.pivot(index=["horizon_s", "skill"], columns="split",
                            values=["class_nll", "f1", "support"])
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(table.round(4))
    print(f"Mean train NLL: {row['train_nll']:.6f}; "
          f"mean test NLL: {row['test_nll']:.6f}; {row['seconds']:.1f} seconds", flush=True)
    if CONFIG["early_stopping"] and row["epoch"] > CONFIG["early_stopping_min_epochs"]:
        print(f"Early-stopping patience: {row['epochs_without_improvement']}/"
              f"{CONFIG['early_stopping_patience']}", flush=True)

if TRAIN_MODEL:
    history = train_model(data, RUN, CONFIG, callback=epoch_output, device=DEVICE, resume=RESUME, tracking=TRACKING)
else:
    assert (RUN / "history.csv").exists(), "Enable TRAIN_MODEL or select a completed run."
    history = pd.read_csv(RUN / "history.csv")
display(history.tail())

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


## Final results from the selected checkpoint

Training can conclude at the early-stopping epoch or at the 100-epoch limit.
The selected checkpoint may precede the final fitted epoch. All final diagnostics below use that
single checkpoint on **train, test and validation**. Validation is first evaluated
here, after selection. A training-frequency baseline provides context; its class
probabilities are fitted only on training labels, separately by horizon.

In [ ]:
%%time
status = read_json(RUN / "training_status.json")
assert status["status"] in {"completed", "early_stopped"}, "Training has not concluded."
assert status["completed_epochs"] == len(history), "Training status/history mismatch."
assert read_json(RUN / "run.json")["training"] == CONFIG, "Run configuration differs."
display(Markdown(f"**Training status: {status['status']}; "
                 f"epochs fitted: {len(history)}/{CONFIG['epochs']}.**"))
best_epoch, final_metrics, final_per_class, final_results = final_evaluation(data, RUN, DEVICE, tracking=TRACKING)
display(Markdown(f"**Selected epoch: {best_epoch}.** Test selected this checkpoint; "
                 "validation was excluded from selection."))
display(final_metrics.round(4))

baseline_rows = []
for split in SPLITS:
    y = data.targets[data.indices[split]]
    probabilities = np.zeros((len(y), 3, len(data.classes)), np.float32)
    for h in range(3):
        train_labels = data.targets[data.indices["train"], h]
        counts = np.bincount(train_labels[train_labels >= 0], minlength=len(data.classes))
        probabilities[:, h] = (counts + 1) / (counts.sum() + len(data.classes))
    baseline, _, _ = metric_tables(y, probabilities, data.classes, split)
    baseline_rows.append(baseline)
baseline_metrics = pd.concat(baseline_rows, ignore_index=True)
baseline_metrics.to_csv(RUN / "training_frequency_baseline.csv", index=False)
display(baseline_metrics.assign(model="Training frequency").round(4))

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


In [ ]:
%%time
DIAGNOSTICS = RUN / "diagnostics"
DIAGNOSTICS.mkdir(exist_ok=True)

def show_figure(fig, name):
    fig.tight_layout()
    fig.savefig(DIAGNOSTICS / f"{name}.png", dpi=160, bbox_inches="tight")
    plt.show()
    plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for split in ["train", "test"]:
    axes[0].plot(history.epoch, history[f"{split}_nll"], label=split)
    axes[1].plot(history.epoch, history[f"{split}_macro_f1"], label=split)
for ax in axes:
    ax.axvline(best_epoch, color="black", linestyle=":", label="Selected epoch")
    ax.set_xlabel("Epoch")
    ax.legend()
axes[0].set(title="Mean horizon loss", ylabel="NLL / cross-entropy")
axes[1].set(title="Mean horizon catalogue macro-F1", ylabel="F1", ylim=(0, 1))
show_figure(fig, "learning_curves")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, measure in zip(axes, ["nll", "accuracy", "macro_f1"]):
    final_metrics.pivot(index="horizon_s", columns="split", values=measure)[list(SPLITS)].plot(
        kind="bar", ax=ax, rot=0)
    ax.set(title=f"Best checkpoint: {measure}", xlabel="Forecast horizon (s)")
    if measure != "nll": ax.set_ylim(0, 1)
show_figure(fig, "final_split_comparison")

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


In [ ]:
%%time
# F1 and support for all catalogue labels, split by horizon.
fig, axes = plt.subplots(2, 3, figsize=(20, 14), sharey=True)
for h, horizon in enumerate(HORIZONS):
    subset = final_per_class.loc[final_per_class.horizon_s.eq(horizon)]
    f1 = subset.pivot(index="skill", columns="split", values="f1").reindex(
        index=data.classes, columns=SPLITS)
    counts = subset.pivot(index="skill", columns="split", values="support").reindex(
        index=data.classes, columns=SPLITS)
    image = axes[0, h].imshow(f1, vmin=0, vmax=1, aspect="auto", cmap="viridis")
    count_image = axes[1, h].imshow(np.log1p(counts), aspect="auto", cmap="magma")
    for ax in axes[:, h]:
        ax.set_xticks(range(3), SPLITS)
        ax.set_yticks(range(len(data.classes)), data.classes, fontsize=7)
    axes[0, h].set_title(f"+{horizon}s per-class F1")
    axes[1, h].set_title(f"+{horizon}s log(1 + support)")
    fig.colorbar(image, ax=axes[0, h], fraction=.04)
    fig.colorbar(count_image, ax=axes[1, h], fraction=.04)
show_figure(fig, "per_class_f1_and_support")

fig, axes = plt.subplots(3, 3, figsize=(24, 23))
for row, split in enumerate(SPLITS):
    for h, horizon in enumerate(HORIZONS):
        cm = final_results[split]["confusion"][h]
        totals = cm.sum(1, keepdims=True)
        normalized = np.divide(cm, totals, out=np.zeros_like(cm, float), where=totals > 0)
        ax = axes[row, h]
        ax.imshow(normalized, vmin=0, vmax=1, cmap="Blues")
        ax.set_xticks(range(len(data.classes)), data.classes, rotation=90, fontsize=6)
        ax.set_yticks(range(len(data.classes)), data.classes, fontsize=6)
        ax.set(title=f"{split}, +{horizon}s", xlabel="Predicted skill", ylabel="True skill")
show_figure(fig, "confusion_matrices")

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


In [ ]:
%%time
# Probability calibration: each point averages one confidence bin.
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for h, horizon in enumerate(HORIZONS):
    axes[h].plot([0, 1], [0, 1], "k:", label="Perfect calibration")
    for split in SPLITS:
        result = final_results[split]
        valid = result["targets"][:, h] >= 0
        p = result["probabilities"][valid, h]
        y = result["targets"][valid, h]
        confidence, correct = p.max(-1), p.argmax(-1) == y
        bins = np.minimum((confidence * 10).astype(int), 9)
        points = [(confidence[bins == b].mean(), correct[bins == b].mean())
                  for b in range(10) if (bins == b).any()]
        axes[h].plot(*np.asarray(points).T, marker="o", label=split)
    axes[h].set(title=f"+{horizon}s reliability", xlabel="Mean confidence",
                ylabel="Observed accuracy", xlim=(0, 1), ylim=(0, 1))
    axes[h].legend()
show_figure(fig, "reliability")

with pd.option_context("display.max_rows", None):
    display(final_per_class.pivot(index=["horizon_s", "skill"], columns="split",
                                  values=["class_nll", "f1", "support"]).round(4))
print("Saved metrics, predictions, checkpoints and PNG diagnostics to:", RUN)
if MLFLOW_ENABLED:
    log_run_artifacts(RUN)  # Attach final figures and baseline tables to the same run.
if (RUN / "mlflow.json").exists():
    print("MLflow run ID:", read_json(RUN / "mlflow.json")["run_id"])

# Include completion of queued GPU kernels in this cell's elapsed wall time.
if torch.cuda.is_available():
    torch.cuda.synchronize()


## Recommendations for further performance improvements

These are proposed experiments, not claims of measured gains. Use the selection
partition for tuning; keep a new independent evaluation set for subsequent work
once the final validation results have been inspected.

1. **Measure the value of pairing.** Compare B01-only, A01-only and paired inputs
   under the same 40/30/30 scenario split. Compare 4 heads at width 128, 4 heads at
   width 192 and 6 heads at width 192 to separate width from head-count effects.
   Paired telemetry can expose A01's own missile state; deploy it only where both
   viewpoints can actually be supplied with the same timing and latency.
2. **Improve rare-class recall.** Inspect per-class F1 alongside support and confusion.
   Try capped inverse-square-root class weights, focal loss or balanced sampling
   fitted only on training labels. Compare unweighted NLL, supported macro-F1 and
   calibration as well as accuracy; absent classes cannot be learned from reweighting.
3. **Model the two viewpoints explicitly.** Compare concatenation with separate
   encoders and cross-attention or aircraft/role embeddings, keeping temporal
   alignment exact. Add shared-world relative geometry only from causal observations.
4. **Reduce overfitting if the curves separate.** Try learning-rate warmup and cosine
   decay, dropout/weight-decay tuning, label smoothing and smaller encoders. Tune
   within a fixed search budget; report variability across several training seeds.
5. **Handle skill changes and equivalent controls.** Diagnose transition windows
   separately from stable labels. Consider hierarchical skill families or a
   multi-task skill/continuous-parameter objective, while keeping target decisions
   out of the feature stream. Distinguish catalogue prediction from manoeuvre recognition.
6. **Calibrate and quantify uncertainty.** If reliability curves show overconfidence,
   fit temperature scaling on a separate calibration subset of the selection data.
   Estimate uncertainty by resampling whole scenario groups, not overlapping windows.
   Report per-pilot performance and leave-one-pilot-out tests for unseen-policy claims.
7. **Stress-test realistic access.** Evaluate missing/delayed viewpoints, sensor noise,
   track loss and new geometries. These corpora provide idealized exact tracks.
   Use structured viewpoint dropout only after defining how missing telemetry is
   represented, and validate it on a matching deployment task.

The loss checkpoint rule follows the requested test-based selection. For a future
experiment, conventional validation-based selection with a fully untouched test set
would make the roles easier to interpret. Do not rename these saved partitions
mid-experiment or compare their scores as if they used notebook 09's original split.

## Training throughput: batching, acceleration estimates and memory

**The accelerated implementation is batched.** The default training batch contains
512 windows of shape `(51, 108)` and produces one optimizer update. The order is
reshuffled every epoch. Evaluation uses batches of 1024 windows under
`torch.inference_mode()`. For the prepared corpus this is **168 optimizer batches**,
**84 train-evaluation batches** and **60 test-evaluation batches** each epoch.
Choosing training batch 1024 reduces optimizer updates to 84 per epoch. The earlier
256/512 baseline used 335/168/120 batches respectively.
The final, smaller batches are retained. Validation is evaluated only at the end.

**Implemented defaults:** BF16 forward passes on supported CUDA GPUs and an FP32
normalized GPU frame cache. This removes repeated CPU normalization and per-batch
host-to-device feature copies. BF16 is resolved to FP32 on CPU/unsupported devices.
**Optional paths:** `DATA_BACKEND="prefetch"` enables persistent workers, pinned
batches and double-buffered nonblocking CUDA copies; `COMPILE_MODEL=True` enables
`torch.compile`. Compilation remains off until benchmarked. Evaluation CE and
probabilities stay FP32. All classes, horizons, masks and test-loss checkpoint rules
are preserved; larger batches can still change optimization/convergence.
CUDA is selected automatically when available. The supplied PyTorch
**2.11.0+cu130** build uses CUDA 13.0 and is consistent with the reported
CUDA-13.1-capable driver; verify actual execution with the setup cell's kernel check.
CPU intra-op threads are now configurable (`CPU_THREADS=4`), capped to the available
logical processors, on the reported **10-core/16-logical-processor Intel CPU**. This limits PyTorch CPU
operations, not the whole process or GPU execution. A 15.5% total CPU utilization
snapshot is consistent with a few active logical processors but does not measure
GPU utilization or prove a bottleneck. Changing this thread count does not create prefetch workers. Those are configured
separately and are bypassed when unique frames are cached on GPU. During epoch evaluation, probability arrays are copied to the CPU after
each batch and retained for both train and test.

**Earlier FP32/CPU-gather baseline measurement:** on this environment (three CPU threads, PyTorch
2.5.1+cpu), a warmed real-data training batch of 256 took **1.110 s** median,
and an evaluation batch of 512 took **0.520 s** median. Each median used three
repetitions after one warmup batch. Training included forward, loss, backward,
gradient clipping and AdamW; evaluation included forward, cross-entropy and
softmax. Warmed CPU window gathering/normalization took about **5 ms per batch**.
These are short batch measurements, not a trained-model or full-epoch benchmark.
Multiplying by the batch counts gives approximately **521 s (8.7 minutes) of
computation per epoch**, before data, checkpoint and notebook-rendering overhead.
Batch tails, cache effects, competing work and GPU execution can change this.
This three-thread CPU environment is not the reported Intel machine. Do not scale
these timings by 10/3 or 16/3: physical/logical cores, memory bandwidth, scheduling,
power limits and CPU/GPU overlap do not scale that way. Family/model and a frequency
snapshot are insufficient to derive a more precise epoch time for the target machine.

**Memory anchor:** the default 976,995-parameter model has **3.73 MiB** of FP32
weights and about **14.91 MiB** for weights, gradients and two Adam moments.
This excludes activations, optimizer temporaries and CUDA overhead. With the accelerated defaults, training input occupies **10.76 MiB** per batch
and evaluation input **21.52 MiB**. The earlier 256/512 baseline used **5.38/10.76 MiB**.
The unique-frame feature file is **1.028 GiB**, memory-mapped rather than eagerly
loaded. Expanding every overlapping window would instead require **4.303 GiB**.
A single layer's FP32 hidden tensor is **19.13 MiB** at batch 512, and its
feed-forward intermediate is **76.50 MiB**; BF16 reduces many such intermediates.
For planning, allow roughly **0.4–1.6 GiB for training activations/workspaces**
at batch 512 in FP32 before BF16 savings, plus model state and framework overhead.
The FP32 GPU cache adds about **1.03 GiB**, plus small anchor/label arrays. This is a broad
architecture-based allowance, not a measured CUDA peak. Kernel choice matters.

The table gives **planning estimates, not measured GPU gains**. No GPU was available
here. The target is the **RTX 5080 (Blackwell), assuming 16 GB VRAM**. Apart from
the first row, speedups are historical planning comparisons against the earlier FP32/CPU-gather
loop on that same GPU (not additional multipliers on top of the enabled accelerations),
using the reported PyTorch 2.11.0+cu130 build after the CUDA kernel check passes,
and assuming that the affected stage is material. The absolute 20–60-second epoch
range remains a broad estimate until measured on the target RTX 5080; the new CPU
information does not turn that estimate into a benchmark.
Each option can give no gain or regress on unsuitable hardware. Ranges overlap:
**do not multiply them** to predict a combined gain. Measure GPU time with CUDA
synchronization and peak memory with `torch.cuda.max_memory_allocated()` and
`torch.cuda.max_memory_reserved()` after resetting peak statistics.

| Recommendation | Estimated whole-epoch speedup | Estimated memory effect | Conditions and tradeoffs |
| --- | --- | --- | --- |
| Run on the RTX 5080 rather than this three-thread CPU | **10–30×** indicative; approximately **20–60 s/epoch** allowing modest host overhead against the 521 s CPU-compute extrapolation. This is an estimate, not an RTX 5080 benchmark | Move model, gradients and activations to VRAM; budget approximately **0.5–2 GiB** total process VRAM for the earlier batch-256/no-cache baseline; roughly **1.5–4 GiB** for accelerated batch 512 plus the frame cache, then measure | Already automatic when CUDA is available. Use the supplied 2.11.0+cu130 build after the kernel check passes. Host CPU, kernels, input transfer and rendering determine the actual gain. |
| Use BF16 autocast on supported GPUs; FP16 with gradient scaling where needed | **1.3–2.0×** | Often **30–50% less activation memory**; FP32 model/Adam state remains about 15 MiB. Roughly **0.12–0.8 GiB** activation savings at batch 512 against the allowance above | Keep reductions, NLL and metric accumulation in FP32. Check loss/F1 and numerical stability. Do not assume this helps the current CPU. |
| Try training batches **512**, then **1024**, and tune evaluation batches separately | **1.05–1.35×** if the GPU is underutilized | Training inputs are **10.76/21.52 MiB**; activation memory is approximately **2×/4×** the earlier batch-256 baseline. Model/Adam state is unchanged | Optimizer updates fall to **168/84 per epoch**; convergence can change. Benchmark throughput and recheck learning-rate/optimization behavior. |
| Cache and normalize unique frames on the GPU once, then gather windows there | **1.05–1.4×** if CPU gathering/transfers are a bottleneck | About **+1.03 GiB VRAM** for FP32 frames, plus roughly 10 MiB of targets/indices; use in-place normalization to avoid another full frame copy | Retain train-only scaling. Cache unique frames, not the 4.30 GiB of expanded windows. A compute-bound GPU may see little gain. |
| If frames stay on CPU, use 2–4 DataLoader workers, pinned buffers and nonblocking transfers with real prefetch overlap | **1.05–1.25×** when the GPU waits for input | Around **43–172 MiB** for 4–8 queued input batches at the new 512/1024 sizes, plus active/pinned buffers and platform-dependent worker overhead (budget roughly **0.1–0.5 GiB per worker**) | Memory maps can share file-backed pages. Avoid duplicating the entire dataset in each worker. On the reported 10-core/16-thread CPU, benchmark 0, 2 and 4 workers first; do not jump to 16. If using Windows, keep Dataset/worker code in an importable module and reopen memory maps per worker to avoid spawn-time dataset copies. This is an alternative to GPU caching. |
| Accumulate epoch confusion counts and per-class NLL on-device; retain probabilities only for final diagnostics | **1.02–1.15×** when evaluation transfers/allocation matter | Removes about **55 MiB** of retained train/test probabilities and up to about **78 MiB** of probability-array peak RAM; summary accumulators are well under 1 MiB | Preserves full per-epoch train/test F1 and loss. Current concatenation temporarily duplicates probability storage. Final reliability plots still need final probabilities. |
| Try `torch.compile` for a fixed-shape training step; benchmark fused AdamW separately | **1.05–1.25×** after compilation; fused AdamW alone is likely only **1.00–1.05×** for this small model | Compiler/graph caches are backend-dependent; reserve roughly **0.1–1 GiB additional RAM/VRAM** and measure. Fused AdamW keeps persistent state size unchanged | Compilation can cost seconds to minutes; batch tails may cause recompilation. Validate outputs and keep the eager path available. |
| For FP32 on Ampere or newer, try `torch.set_float32_matmul_precision("high")` / TF32 | **1.05–1.4×** when matrix multiplication dominates | Approximately **no change** in persistent or activation storage | Lower internal multiplication precision; check metrics. This overlaps BF16/Tensor Core gains and should not be counted as an independent multiplier. |
| **Only if the per-epoch full-training-metrics requirement is relaxed:** evaluate the full train split every five epochs; keep test evaluation every epoch | About **1.20× averaged over five epochs**, assuming backward costs about twice forward | Little change in peak memory on epochs that still evaluate train; fewer large train probability allocations on other epochs | This changes the requested reporting schedule, so it is **not enabled**. Online training-batch metrics use changing weights/dropout and are not an equivalent replacement. |

For the last row, the estimate uses forward-work units: current cost is
`3*N_train + N_train + N_test`; the proposed average is
`3*N_train + N_train/5 + N_test`, giving **1.205×** for this split. Actual training
and inference kernels differ. No recommendation changes the held-out validation
policy or the requested test-loss checkpoint rule.

On the RTX 5080, first verify the PyTorch CUDA build and BF16 support, then
benchmark batch 512 and 1024. The 1.03 GiB frame cache should fit comfortably in
16 GB at the present model size, subject to measured activation/workspace peaks.
Start with GPU execution and BF16; then profile input transfer,
evaluation, checkpoint writes and notebook rendering before choosing caching,
prefetch or compilation. This model is small, so data movement and Python/kernel
launch overhead can limit the gains from faster matrix operations.

**CPU-specific recommendation:** if CPU model execution is required, benchmark
**3, 4 and 8** with the configurable `CPU_THREADS` limit
intra-op threads on the target Intel CPU. A planning range is **1.0–1.5×** for a
CPU-only compute-bound epoch relative to that same CPU with three threads; memory
bandwidth and core scheduling can erase the gain. Allow roughly **10–100 MiB**
additional CPU scratch/stack memory as a broad, backend-dependent budget and measure
RSS. For GPU training expect only **1.0–1.1×**, often no gain, from this change alone.
Leave inter-op settings unchanged for that comparison. GPU input throughput is more
likely to benefit from frame caching or a small prefetch pool than from increasing
PyTorch intra-op threads. The 15.5% utilization snapshot is not a reason to occupy
all 16 logical processors. The default requested limit is now four, capped to the executing machine's logical
processor count. Keep it fixed while comparing GPU acceleration options.



## Compare tracked runs

This view uses runs in the configured experiment with the same prepared-data hash.
It displays final held-out scores alongside test-selected model quality. Curves use
training/test history only. The saved model URI can be loaded with
`mlflow.pytorch.load_model` after setting the matching tracking URI, or by downloading
its model artifact through `MlflowClient`.


In [ ]:
%%time
if MLFLOW_ENABLED:
    comparison = compare_runs(TRACKING, sha256(DATA / "manifest.json"))
    if comparison.empty:
        print("No tracked runs for this prepared dataset yet.")
    else:
        with pd.option_context("display.max_columns", None):
            display(comparison.sort_values("best_test_nll", na_position="last"))
        comparison.to_csv(RUN / "mlflow_comparison.csv", index=False)
        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        for row in comparison.head(5).itertuples():
            for ax, split in zip(axes, ("train", "test")):
                curve = metric_history(TRACKING, row.run_id, f"epoch/{split}/nll")
                if not curve.empty:
                    ax.plot(curve.epoch, curve.value, label=str(row.name)[:55])
                ax.set(title=f"Tracked {split} loss", xlabel="Epoch", ylabel="NLL")
        for ax in axes:
            handles, labels = ax.get_legend_handles_labels()
            if handles:
                ax.legend(fontsize=7)
        show_figure(fig, "mlflow_run_comparison")
        log_run_artifacts(RUN)
else:
    print("MLflow comparison disabled.")

if torch.cuda.is_available():
    torch.cuda.synchronize()
